# GetDataset

In [ ]:
#Install Libraries
!pip install -q transformers datasets evaluate torchaudio accelerate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 56.4 MB/s eta 0:00:00


In [ ]:
from datasets import load_from_disk, Audio
from transformers import AutoProcessor, AutoModelForCTC, TrainingArguments, Trainer
import evaluate
import numpy as np
import torch

In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"nalaprogroup","key":"6e69ae1b1b69e1a4d23d6edf0680d84f"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d nalaprogroup/data-nalapro-project03

Dataset URL: https://www.kaggle.com/datasets/nalaprogroup/data-nalapro-project03
License(s): CC0-1.0
100% 354M/354M [00:25<00:00, 14.8MB/s]



In [ ]:
import zipfile

zip_path = "data-nalapro-project03.zip"
extract_path = "/content/sample_data/data-nalapro-project03"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted to:", extract_path)

Extracted to: /content/sample_data/data-nalapro-project03


In [ ]:
print("Loading dataset...")
dataset = load_from_disk(extract_path)
dataset

Loading dataset...


DatasetDict({
    train: Dataset({
        features: ['audio', 'intent_class', 'transcription'],
        num_rows: 1800
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 56
    })
    test: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 57
    })
})

In [ ]:
# Variable kan dataset
train_dataset = dataset["train"]
dev_dataset = dataset["validation"]
test_dataset = dataset["test"]

In [ ]:
from datasets import DatasetDict


# =========================
# FINAL DATASET
# =========================
final_dataset = DatasetDict({
    "train": train_dataset,
    "validation": dev_dataset,
    "test": test_dataset
})

print(final_dataset)

DatasetDict({
    train: Dataset({
        features: ['audio', 'intent_class', 'transcription'],
        num_rows: 1800
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 56
    })
    test: Dataset({
        features: ['audio', 'transcription', 'intent_class'],
        num_rows: 57
    })
})


In [ ]:
print("Train:", len(final_dataset["train"]))
print("Dev:", len(final_dataset["validation"]))
print("Test:", len(final_dataset["test"]))

Train: 1800
Dev: 56
Test: 57


#Prerocessing
Konsep Dasar Whisper :
    Audio → Log-Mel Spectrogram → Encoder → Decoder → Text


In [ ]:
# 1. Preprocessing (ASR Version)
from transformers import WhisperProcessor

# Processor menggabungkan Feature Extractor (audio) dan Tokenizer (teks)
processor = WhisperProcessor.from_pretrained("openai/whisper-base")

def prepare_dataset_asr(batch):
    # 1. Proses Audio (Input)
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=16000
    ).input_features[0]

    # 2. Proses Teks (Label/Target)
    # Tokenisasi teks transkripsi menjadi ID label
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

# Jalankan mapping
encoded_dataset = final_dataset.map(
    prepare_dataset_asr,
    remove_columns=final_dataset["train"].column_names,
    num_proc=1
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/56 [00:00<?, ? examples/s]

Map:   0%|          | 0/57 [00:00<?, ? examples/s]

In [ ]:
# Data Collator
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Padding input audio
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Padding label teks
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Ganti padding token ID dengan -100 agar tidak dihitung di loss function
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
#pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.3 MB/s eta 0:00:00


In [ ]:
# Evaluasi Menggunakan WER
import evaluate

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Ganti -100 agar bisa di-decode
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

In [ ]:
#Load Dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Training Arguments & Model
from transformers import WhisperForConditionalGeneration, TrainingArguments, Trainer, GenerationConfig
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")

# Buat konfigurasi generasi yang bersih
generation_config = GenerationConfig.from_pretrained("openai/whisper-base")
generation_config.suppress_tokens = []
generation_config.forced_decoder_ids = None

# Masukkan kembali ke model
model.generation_config = generation_config

# Konfigurasi model
#model.config.forced_decoder_ids = None
# model.config.suppress_tokens = []

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/Model/whisper-asr-minds14-01-Augm",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    num_train_epochs=5,
    max_steps=-1,
    warmup_steps=500,
    fp16=True,

    # --- PERBAIKAN DI SINI ---
    eval_strategy="epoch",       # Evaluasi setiap akhir epoch agar pasti muncul
    save_strategy="epoch",       # Simpan model setiap akhir epoch
    logging_steps=10,            # Munculkan Training Loss setiap 10 langkah

    predict_with_generate=True,
    generation_max_length=225,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to="none",

    # Tambahkan ini untuk memastikan log muncul di notebook
    disable_tqdm=False,
    load_best_model_at_end=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
    #tokenizer=processor.feature_extractor,
)

# Cek apakah fungsi evaluasi berjalan sebelum training berat dimulai
print(trainer.evaluate())

trainer.train()

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, 

{'eval_loss': 5.207865238189697, 'eval_model_preparation_time': 0.0039, 'eval_wer': 41.838134430727024, 'eval_runtime': 21.2763, 'eval_samples_per_second': 2.632, 'eval_steps_per_second': 0.329}


Epoch,Training Loss,Validation Loss,Model Preparation Time,Wer
1,2.838888,0.849503,0.003900,33.470508
2,0.868385,0.394215,0.003900,28.532236
3,0.361072,0.398817,0.003900,27.709191
4,0.062135,0.452839,0.003900,27.983539
5,0.018753,0.498675,0.003900,27.297668


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=565, training_loss=1.7353095886849723, metrics={'train_runtime': 1575.8698, 'train_samples_per_second': 5.711, 'train_steps_per_second': 0.359, 'total_flos': 5.8373996544e+17, 'train_loss': 1.7353095886849723, 'epoch': 5.0})

In [ ]:
# Simpan hasil akhir model dan processor agar bisa dipakai lagi nanti
trainer.save_model("/content/drive/MyDrive/Colab Notebooks/Model/whisper-asr-minds14-Augm-FINAL")
processor.save_pretrained("/content/drive/MyDrive/Colab Notebooks/Model/whisper-asr-minds14-Augm-FINAL")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['/content/drive/MyDrive/Colab Notebooks/Model/whisper-asr-minds14-Augm-FINAL/processor_config.json']

In [ ]:
# TESTING INFERENCE
# 1. Cara Instan (Hugging Face Pipeline)
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from IPython.display import Audio, display

model_path = "/content/drive/MyDrive/Colab Notebooks/Model/whisper-asr-minds14-Augm-FINAL"

# 1. Load Model & Processor
model = WhisperForConditionalGeneration.from_pretrained(model_path).to("cuda")
processor = WhisperProcessor.from_pretrained(model_path)

# 2. Ambil sampel (misalnya indeks ke-0 dari data test)
sample = final_dataset["test"][1]
audio_array = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]

# 3. Tampilkan Pemutar Audio
print("Memutar Audio Asli:")
display(Audio(audio_array, rate=sampling_rate))

# 4. Proses Prediksi (Manual Inference)
input_features = processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt").input_features.to("cuda")

with torch.no_grad():
    predicted_ids = model.generate(input_features)

# 5. Decode dan Tampilkan Hasil Teks
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print(f"\n{'='*30}")
print(f"Hasil Prediksi : {transcription}")
print(f"Teks Sebenarnya: {sample['transcription']}")
print(f"{'='*30}")

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Memutar Audio Asli:



Hasil Prediksi :  How do I set up a joint account
Teks Sebenarnya: how do I set up a joint account


In [ ]:
# 2. Cara Evaluasi Massal (Menguji Seluruh Test Set)
# Jalankan prediksi pada seluruh encoded_test
test_predictions = trainer.predict(encoded_dataset["test"])

# Ambil metriknya (termasuk WER)
print("Metrik pada Test Set:")
print(test_predictions.metrics)

Metrik pada Test Set:
{'test_loss': 0.47102436423301697, 'test_model_preparation_time': 0.0039, 'test_wer': 27.225130890052355, 'test_runtime': 12.1377, 'test_samples_per_second': 4.696, 'test_steps_per_second': 0.659}


In [ ]:
# Cara Manual (Input Audio Milik Sendiri)
import torch
import librosa
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from IPython.display import Audio, display

# 1. Path ke model dan file audio Anda
model_path = "/content/drive/MyDrive/Colab Notebooks/Model/whisper-asr-minds14-Augm-FINAL"
path_audio_anda = "/content/sample_data/test_01.wav"  # Ganti dengan nama file yang Anda upload

# 2. Load Model & Processor
model = WhisperForConditionalGeneration.from_pretrained(model_path).to("cuda")
processor = WhisperProcessor.from_pretrained(model_path)

# 3. Load dan Resample Audio ke 16kHz
# librosa.load otomatis mengubah audio ke mono dan sampling rate yang kita tentukan
audio_input, sr = librosa.load(path_audio_anda, sr=16000)

# 4. Tampilkan Pemutar Audio untuk verifikasi
print("Mendengarkan file yang diuji:")
display(Audio(audio_input, rate=sr))

# 5. Preprocessing
input_features = processor(audio_input, sampling_rate=sr, return_tensors="pt").input_features.to("cuda")

# 6. Prediksi (Inference)
with torch.no_grad():
    predicted_ids = model.generate(input_features)

# 7. Decode Hasil
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

print(f"\n{'='*40}")
print(f"HASIL TRANSKRIPSI: {transcription}")
print(f"{'='*40}")

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Mendengarkan file yang diuji:



HASIL TRANSKRIPSI:  hello morning I won't check my account can you help
